In [1]:
import pandas as pd
import numpy as np
from scipy import stats

In [5]:
def convert_volume(v):
    v = str(v).strip()
    if 'M' in v:
        return float(v.replace('M', '')) * 1000
    elif 'K' in v:
        return float(v.replace('K', ''))
    else:
        return float(v)

df = pd.read_csv("data/RTX.csv")
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date').reset_index(drop=True)
df['Vol.'] = df['Vol.'].apply(convert_volume)
df['Change %'] = df['Change %'].str.replace('%', '').astype(float) / 100
df['return'] = df['Price'].pct_change()
df['ticker'] = 'RTX'

df2 = pd.read_csv("data/LMT.csv")
df2['Date'] = pd.to_datetime(df2['Date'])
df2 = df2.sort_values('Date').reset_index(drop=True)
df2['Vol.'] = df2['Vol.'].apply(convert_volume)
df2['Change %'] = df2['Change %'].str.replace('%', '').astype(float) / 100
df2['return'] = df2['Price'].pct_change()
df2['ticker'] = 'LMT'

df3 = pd.read_csv("data/NOC.csv")
df3['Date'] = pd.to_datetime(df3['Date'])
df3 = df3.sort_values('Date').reset_index(drop=True)
df3['Vol.'] = df3['Vol.'].apply(convert_volume)
df3['Change %'] = df3['Change %'].str.replace('%', '').astype(float) / 100
df3['return'] = df3['Price'].pct_change()
df3['ticker'] = 'NOC'

combined = pd.concat([df, df2, df3], ignore_index=True)
combined = combined.sort_values(['ticker', 'Date']).reset_index(drop=True)

In [7]:
combined['vol_avg20'] = combined.groupby('ticker')['Vol.'].transform(lambda x: x.rolling(20).mean())
combined['vol_ratio'] = combined['Vol.'] / combined['vol_avg20']
combined['vol_ratio_pctile'] = combined.groupby('ticker')['vol_ratio'].rank(pct=True)
combined['high_volume'] = combined['vol_ratio_pctile'] >= 0.70

In [9]:
train = combined[(combined['Date'] >= '2010-01-01') & (combined['Date'] < '2020-01-01')].reset_index(drop=True)
test = combined[(combined['Date'] >= '2020-01-01') & (combined['Date'] <= '2026-12-31')].reset_index(drop=True)

In [11]:
thresholds = [0.02, 0.05, 0.07, 0.10, 0.12, 0.15]
holding_periods = [1, 2, 3, 4, 5, 7, 10, 15]

In [13]:
def run_grid(data, thresholds, holding_periods):
    results = []
    for t in thresholds:
        for h in holding_periods:
            signal_returns = []
            baseline_returns = []
            for ticker in data['ticker'].unique():
                sub = data[data['ticker'] == ticker].reset_index(drop=True)
                signal_days = sub.index[sub['return'] <= -t].tolist()
                other_days = sub.index[sub['return'] > -t].tolist()

                for i in signal_days:
                    if i + h < len(sub):
                        fwd_ret = (sub['Price'].iloc[i + h] - sub['Price'].iloc[i]) / sub['Price'].iloc[i]
                        signal_returns.append(fwd_ret)

                for i in other_days:
                    if i + h < len(sub):
                        fwd_ret = (sub['Price'].iloc[i + h] - sub['Price'].iloc[i]) / sub['Price'].iloc[i]
                        baseline_returns.append(fwd_ret)

            if len(signal_returns) > 1 and len(baseline_returns) > 1:
                win_rate = sum([1 for r in signal_returns if r > 0]) / len(signal_returns)
                avg_return = sum(signal_returns) / len(signal_returns)
                n = len(signal_returns)
                t_stat, p_value = stats.ttest_ind(signal_returns, baseline_returns, equal_var=False)
            else:
                win_rate = None
                avg_return = None
                n = len(signal_returns)
                t_stat = None
                p_value = None

            results.append({
                'threshold': t,
                'holding_days': h,
                'win_rate': win_rate,
                'avg_return': avg_return,
                'n_obs': n,
                't_stat': t_stat,
                'p_value': p_value
            })
    return pd.DataFrame(results)

In [15]:
train_results = run_grid(train, thresholds, holding_periods)
test_results = run_grid(test, thresholds, holding_periods)

In [17]:
train_winrate = train_results.pivot(index='threshold', columns='holding_days', values='win_rate')
train_tstat = train_results.pivot(index='threshold', columns='holding_days', values='t_stat')
train_pvalue = train_results.pivot(index='threshold', columns='holding_days', values='p_value')
train_n = train_results.pivot(index='threshold', columns='holding_days', values='n_obs')

test_winrate = test_results.pivot(index='threshold', columns='holding_days', values='win_rate')
test_tstat = test_results.pivot(index='threshold', columns='holding_days', values='t_stat')
test_pvalue = test_results.pivot(index='threshold', columns='holding_days', values='p_value')
test_n = test_results.pivot(index='threshold', columns='holding_days', values='n_obs')

print(train_winrate)
print(train_tstat)
print(train_pvalue)
print(train_n)



holding_days        1         2         3         4         5     7   \
threshold                                                              
0.02          0.538462  0.532308  0.535385  0.603077  0.612308  0.60   
0.05          0.550000  0.450000  0.500000  0.650000  0.750000  0.65   
0.07          0.500000  0.500000  0.500000  0.500000  0.500000  0.50   
0.10               NaN       NaN       NaN       NaN       NaN   NaN   
0.12               NaN       NaN       NaN       NaN       NaN   NaN   
0.15               NaN       NaN       NaN       NaN       NaN   NaN   

holding_days        10        15  
threshold                         
0.02          0.632716  0.651235  
0.05          0.650000  0.750000  
0.07          0.500000  0.500000  
0.10               NaN       NaN  
0.12               NaN       NaN  
0.15               NaN       NaN  
holding_days        1         2         3         4         5         7   \
threshold                                                          

In [19]:
print(test_winrate)
print(test_tstat)
print(test_pvalue)
print(test_n)

holding_days        1         2         3         4         5         7   \
threshold                                                                  
0.02          0.489971  0.531792  0.575581  0.584302  0.596491  0.608187   
0.05          0.490196  0.450980  0.568627  0.568627  0.549020  0.607843   
0.07          0.620690  0.448276  0.586207  0.551724  0.551724  0.655172   
0.10          0.777778  0.444444  0.777778  0.777778  0.555556  0.777778   
0.12          1.000000  0.400000  0.600000  0.800000  0.600000  0.800000   
0.15               NaN       NaN       NaN       NaN       NaN       NaN   

holding_days        10        15  
threshold                         
0.02          0.616959  0.581871  
0.05          0.627451  0.607843  
0.07          0.724138  0.689655  
0.10          0.888889  0.888889  
0.12          1.000000  1.000000  
0.15               NaN       NaN  
holding_days        1         2         3         4         5         7   \
threshold                          

In [21]:
from statsmodels.stats.multitest import multipletests

train_results['p_corrected'] = multipletests(train_results['p_value'].fillna(1), method='bonferroni')[1]
test_results['p_corrected'] = multipletests(test_results['p_value'].fillna(1), method='bonferroni')[1]

train_sig = train_results[train_results['p_corrected'] < 0.05]
test_sig = test_results[test_results['p_corrected'] < 0.05]

print(train_sig[['threshold', 'holding_days', 'p_corrected', 'n_obs']])
print(test_sig[['threshold', 'holding_days', 'p_corrected', 'n_obs']])

overlap = pd.merge(train_sig, test_sig, on=['threshold', 'holding_days'], suffixes=('_train', '_test'))
print(overlap[['threshold', 'holding_days', 'p_corrected_train', 'p_corrected_test', 'n_obs_train', 'n_obs_test']])

Empty DataFrame
Columns: [threshold, holding_days, p_corrected, n_obs]
Index: []
   threshold  holding_days  p_corrected  n_obs
3       0.02             4     0.045279    344
6       0.02            10     0.013462    342
7       0.02            15     0.025524    342
Empty DataFrame
Columns: [threshold, holding_days, p_corrected_train, p_corrected_test, n_obs_train, n_obs_test]
Index: []
